# GFE-Net Fine-Tuning — Itapecuru Dataset (Phelcom Eyer)

Este notebook realiza o **fine-tuning** do modelo GFE-Net a partir dos pesos pré-treinados do autor original,
usando imagens de retinografia do dataset Itapecuru capturadas com câmera Phelcom Eyer.

## Requisitos
- **GPU**: T4 ou P100 (Kaggle gratuito)
- **Dataset Kaggle**: Pacote ZIP com o repositório completo + dados preparados
- **Tempo estimado**: ~2-4 horas para 40 epochs (dependendo da GPU e qtd de dados)

## 1. Setup — Descompactar e Instalar Dependências

Ajuste o caminho do ZIP conforme o slug do seu Kaggle Dataset.

In [ ]:
import os

# ========== CONFIGURAÇÕES — AJUSTE AQUI ==========
# Slug do dataset no Kaggle (ex: 'seuusuario/afe-gfenet-finetune')
KAGGLE_DATASET_SLUG = 'seuusuario/afe-gfenet-finetune'  # <-- ALTERE AQUI
ZIP_FILENAME = 'AFE_finetune.zip'  # Nome do arquivo ZIP no dataset

# Caminhos derivados
KAGGLE_INPUT = f'/kaggle/input/{KAGGLE_DATASET_SLUG.split("/")[-1]}'
PROJECT_DIR = '/kaggle/working/project/Annotation-free-Fundus-Image-Enhancement'
# =================================================

In [ ]:
# Descompactar o projeto
!mkdir -p /kaggle/working/project
!unzip -q {KAGGLE_INPUT}/{ZIP_FILENAME} -d /kaggle/working/project

# Navegar para o diretório do projeto
%cd {PROJECT_DIR}

# Instalar dependências
!pip install -q -r requirements.txt

In [ ]:
# Verificar estrutura do dataset de treino
import os

dataroot = os.path.join(PROJECT_DIR, 'datasets', 'gfenet_finetune')
for sub in ['source', 'source_mask', 'target', 'target_mask']:
    path = os.path.join(dataroot, sub)
    count = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f'{sub}/: {count} arquivos')

# Verificar pesos pré-treinados
ckpt_path = os.path.join(PROJECT_DIR, 'checkpoints', 'gfenet', 'latest_net_G.pth')
assert os.path.isfile(ckpt_path), f'Pesos não encontrados: {ckpt_path}'
ckpt_size = os.path.getsize(ckpt_path) / 1024 / 1024
print(f'\nCheckpoint: {ckpt_path} ({ckpt_size:.1f} MB) ✓')

## 2. (Opcional) Preparar Dados no Kaggle

Se você já preparou os dados localmente com `scripts/prepare_gfenet_training.py` e incluiu no ZIP,
**pule esta célula**.

Caso contrário, execute a preparação aqui (demora ~15-30 min para ~364 imagens × 8 degradações).

In [ ]:
# DESCOMENTE SE PRECISAR GERAR OS DADOS DE TREINO NO KAGGLE
#
# !python scripts/prepare_gfenet_training.py \
#     --csv itapecuru_all_labeled_v3_com_tipo.csv \
#     --images_dir ORIGINAL_CLEAN_768x768 \
#     --output_dir datasets/gfenet_finetune \
#     --num_degradations 8 \
#     --image_size 768

## 3. Copiar Pesos para Pasta do Fine-Tuning

O treinamento usa `--name gfenet_finetune` para não sobrescrever os pesos originais.
Copiamos os pesos pré-treinados para a nova pasta de checkpoints.

In [ ]:
import shutil

# Criar pasta de checkpoints para fine-tuning
finetune_ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints', 'gfenet_finetune')
os.makedirs(finetune_ckpt_dir, exist_ok=True)

# Copiar pesos originais
src = os.path.join(PROJECT_DIR, 'checkpoints', 'gfenet', 'latest_net_G.pth')
dst = os.path.join(finetune_ckpt_dir, 'latest_net_G.pth')
shutil.copy2(src, dst)
print(f'Pesos copiados: {dst}')
print(f'Tamanho: {os.path.getsize(dst)/1024/1024:.1f} MB')

## 4. Fine-Tuning

### Hiperparâmetros
| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `--lr` | 0.0005 | 4× menor que o original (0.002) para preservar features pré-aprendidas |
| `--n_epochs` | 30 | Epochs com LR constante |
| `--n_epochs_decay` | 10 | Epochs com LR decaindo linearmente até 0 |
| `--batch_size` | 4 | Ajuste para VRAM da T4 (16GB) com imagens 768×768 |
| `--load_size / --crop_size` | 768 | Resolução nativa das imagens Phelcom |
| `--continue_train` | - | Carrega `latest_net_G.pth` (pesos do autor) |
| `--display_id -1` | - | Desabilita Visdom (não funciona no Kaggle) |

> Se ocorrer **OOM (Out of Memory)**, reduza `--batch_size` para 2 ou 1.

In [ ]:
# ========== HIPERPARÂMETROS — AJUSTE SE NECESSÁRIO ==========
BATCH_SIZE = 4       # Reduzir para 2 ou 1 se der OOM
LR = 0.0005          # Learning rate (conservador para fine-tuning)
N_EPOCHS = 30        # Epochs com LR constante
N_EPOCHS_DECAY = 10  # Epochs com decay linear do LR
SAVE_FREQ = 10       # Salvar checkpoint a cada N epochs
IMG_SIZE = 768       # Resolução das imagens
# ============================================================

train_cmd = f"""python train.py \
    --dataroot datasets/gfenet_finetune \
    --name gfenet_finetune \
    --model gfenet \
    --dataset_mode cataract_with_mask \
    --direction AtoB \
    --norm instance \
    --input_nc 6 \
    --load_size {IMG_SIZE} --crop_size {IMG_SIZE} \
    --batch_size {BATCH_SIZE} \
    --gpu_ids 0 \
    --lr {LR} \
    --lr_policy linear \
    --n_epochs {N_EPOCHS} \
    --n_epochs_decay {N_EPOCHS_DECAY} \
    --save_epoch_freq {SAVE_FREQ} \
    --continue_train \
    --epoch latest \
    --display_id -1 \
    --no_flip
"""

print('Comando de treinamento:')
print(train_cmd)

In [ ]:
import subprocess
import shlex

# Executar o fine-tuning
ret = os.system(train_cmd)
if ret != 0:
    print(f'\n⚠ Treinamento terminou com código {ret}')
else:
    print('\n✓ Treinamento concluído com sucesso!')

## 5. Verificar Resultados do Treinamento

Checar se os checkpoints foram salvos e as losses convergiram.

In [ ]:
# Listar checkpoints salvos
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints', 'gfenet_finetune')
if os.path.isdir(ckpt_dir):
    files = sorted(os.listdir(ckpt_dir))
    print('Checkpoints salvos:')
    for f in files:
        size = os.path.getsize(os.path.join(ckpt_dir, f)) / 1024 / 1024
        print(f'  {f} ({size:.1f} MB)')
else:
    print('Nenhum checkpoint encontrado!')

In [ ]:
# Visualizar curva de loss (se o log existe)
loss_log = os.path.join(ckpt_dir, 'loss_log.txt')
if os.path.isfile(loss_log):
    with open(loss_log) as f:
        lines = f.readlines()
    # Mostrar últimas 20 linhas
    print('Últimas linhas do log de loss:')
    for line in lines[-20:]:
        print(line.strip())
else:
    print('Log de loss não encontrado (pode estar em outro local)')

## 6. Inferência com o Modelo Fine-Tunado

Rodar a inferência nas imagens `quality=2` (rejeitadas) para avaliar a qualidade da restauração.

In [ ]:
# Inferência com modelo fine-tunado
test_ft_cmd = f"""python test.py \
    --dataroot datasets/gfenet_finetune \
    --name gfenet_finetune \
    --model gfenet \
    --dataset_mode cataract_with_mask \
    --load_size {IMG_SIZE} --crop_size {IMG_SIZE} \
    --results_dir /kaggle/working/results_finetune \
    --num_test 100000 \
    --eval"""
os.system(test_ft_cmd)

In [ ]:
# Inferência com modelo ORIGINAL (para comparação)
test_orig_cmd = f"""python test.py \
    --dataroot datasets/gfenet_finetune \
    --name gfenet \
    --model gfenet \
    --dataset_mode cataract_with_mask \
    --load_size {IMG_SIZE} --crop_size {IMG_SIZE} \
    --results_dir /kaggle/working/results_original \
    --num_test 100000 \
    --eval"""
os.system(test_orig_cmd)

## 7. Comparação Visual

Compare as saídas do modelo original vs. fine-tunado lado a lado.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

def show_comparison(original_dir, finetune_dir, n_samples=5):
    """Mostra comparação entre modelo original e fine-tunado."""
    # Buscar imagens fake_TB (restauração RGB) do finetune
    ft_images = sorted(glob.glob(os.path.join(finetune_dir, '**', '*fake_TB*'), recursive=True))[:n_samples]
    
    if not ft_images:
        print('Nenhuma imagem de resultado encontrada!')
        return
    
    fig, axes = plt.subplots(len(ft_images), 3, figsize=(18, 6 * len(ft_images)))
    if len(ft_images) == 1:
        axes = axes[None, :]
    
    for i, ft_path in enumerate(ft_images):
        basename = os.path.basename(ft_path)
        
        # Input (real_TA)
        input_name = basename.replace('fake_TB', 'real_TA')
        input_path = os.path.join(os.path.dirname(ft_path), input_name)
        
        # Original model result
        orig_path = ft_path.replace(finetune_dir, original_dir)
        
        if os.path.isfile(input_path):
            axes[i, 0].imshow(Image.open(input_path))
            axes[i, 0].set_title('Input (Degradada)', fontsize=12)
        axes[i, 0].axis('off')
        
        if os.path.isfile(orig_path):
            axes[i, 1].imshow(Image.open(orig_path))
            axes[i, 1].set_title('Modelo Original', fontsize=12)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(Image.open(ft_path))
        axes[i, 2].set_title('Fine-Tunado', fontsize=12)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

show_comparison(
    '/kaggle/working/results_original/gfenet/test_latest/images',
    '/kaggle/working/results_finetune/gfenet_finetune/test_latest/images',
    n_samples=5
)

## 8. Exportar Resultados e Pesos Fine-Tunados

In [ ]:
import shutil

# Copiar pesos fine-tunados para o diretório de output
output_dir = '/kaggle/working/output'
os.makedirs(output_dir, exist_ok=True)

# Copiar todos os checkpoints .pth
for f in os.listdir(ckpt_dir):
    if f.endswith('.pth'):
        shutil.copy2(
            os.path.join(ckpt_dir, f),
            os.path.join(output_dir, f)
        )
        print(f'Exportado: {f}')

# Comprimir resultados
os.system('cd /kaggle/working && zip -r -q outputs.zip output/ results_finetune/ results_original/')
print('\nArquivo outputs.zip criado em /kaggle/working/')
print('Baixe pela aba Output do Kaggle.')

## 9. (Opcional) Usar os Pesos Fine-Tunados para Inferência no Futuro

Após baixar os pesos `.pth` exportados:

1. Coloque o arquivo `latest_net_G.pth` fine-tunado em `checkpoints/gfenet_finetune/`
2. Execute a inferência:

```bash
python test.py \
    --dataroot datasets/my_gfenet_eval \
    --name gfenet_finetune \
    --model gfenet \
    --dataset_mode cataract_with_mask \
    --load_size 768 --crop_size 768 \
    --results_dir ./results \
    --num_test 100000 \
    --preserve_subfolders --eval
```